# Accessibility and Validation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rahulkaushal04/plotstyle/blob/main/examples/notebooks/02_accessibility_and_validation.ipynb)

A good scientific figure should be readable by everyone, including people with color blindness and readers of black-and-white printed journals.

This notebook shows you how to check both, and how to validate a figure against journal submission rules.

**What you will learn:**
- Preview how your figure looks to someone with color blindness using `plotstyle.preview_colorblind()`
- Preview how your figure looks in grayscale using `plotstyle.preview_grayscale()`
- Check programmatically whether your colors are safe for grayscale printing
- Run a full validation check with `style.validate()`
- Read and use the individual check results

In [ ]:
# Install plotstyle if needed (e.g., on Google Colab).
try:
    import plotstyle
except ImportError:
    import subprocess
    import sys

    subprocess.check_call([sys.executable, "-m", "pip", "install", "plotstyle[color]"])
    import plotstyle

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import plotstyle

## 1. Build a Sample Figure

First, let's create a figure with multiple colored lines. This is the kind of plot where accessibility matters most, since readers need to tell the lines apart.

In [ ]:
x = np.linspace(0, 8, 100)

with plotstyle.use("nature") as style:
    colors = style.palette(n=4)
    fig, ax = style.figure(columns=1)

    for i, c in enumerate(colors):
        ax.plot(x, np.sin(x + i * 0.7), color=c, linewidth=1.5, label=f"Condition {i + 1}")

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Signal (a.u.)")
    ax.legend()
    plt.show()

# fig is still accessible after the with block for previews below

## 2. Colorblind Simulation

About 8% of men have some form of color blindness. `plotstyle.preview_colorblind(fig)` shows you how your figure looks to people with three common types:

| Type | What is affected | How common |
|---|---|---|
| **Deuteranopia** | Cannot distinguish green | ~6% of males |
| **Protanopia** | Cannot distinguish red | ~2% of males |
| **Tritanopia** | Cannot distinguish blue | very rare |

The output is a side-by-side comparison: your original figure plus one panel for each type.

In [ ]:
# Side-by-side: original + deuteranopia + protanopia + tritanopia
cvd_fig = plotstyle.preview_colorblind(fig)
plt.show()
plt.close(cvd_fig)

## 3. Grayscale Preview

Some journals (like IEEE) print figures in black and white. `plotstyle.preview_grayscale(fig)` shows your figure as it would appear in print.

If data series look the same shade of gray, readers will not be able to tell them apart. In that case, consider adding different line styles or markers. You can get these automatically with `plotstyle.palette(..., with_markers=True)`.

In [ ]:
# Side-by-side: original + grayscale rendering
gray_fig = plotstyle.preview_grayscale(fig)
plt.show()
plt.close(gray_fig)
plt.close(fig)

## 4. Check Grayscale Safety in Code

Instead of visually inspecting a preview, you can check programmatically whether your colors are distinct enough when printed in black and white.

Two helper functions make this easy:
- `is_grayscale_safe(colors)`: returns `True` if all colors are sufficiently different in brightness.
- `luminance_delta(colors)`: returns how different each pair of colors is in brightness, sorted from most similar to most different.

In [ ]:
from plotstyle.color.grayscale import is_grayscale_safe, luminance_delta

palette_colors = plotstyle.palette("nature", n=4)

# A threshold of 0.10 means every pair must differ by at least 10% of the luminance scale.
safe = is_grayscale_safe(palette_colors, threshold=0.1)
print(f"Nature palette grayscale-safe (threshold=0.1): {safe}")

# Inspect individual pair deltas, sorted ascending so the weakest pair is first.
deltas = luminance_delta(palette_colors)
print("\nPairwise luminance deltas (sorted ascending):")
for i, j, delta in deltas:
    status = "pass" if delta >= 0.1 else "fail"
    print(f"  [{status}]  Color {i} vs Color {j}: delta = {delta:.3f}")

## 5. Run a Full Validation

`style.validate(fig)` runs all checks in one go: dimensions, fonts, colors, and export settings. Each check results in a PASS, FAIL, or WARN.

In [ ]:
rng = np.random.default_rng(0)

with plotstyle.use("nature") as style:
    fig, ax = style.figure(columns=1)
    colors = style.palette(n=3)

    for i, c in enumerate(colors):
        ax.plot(np.arange(20), rng.normal(i, 0.5, 20), color=c, label=f"Group {i + 1}")

    ax.set_xlabel("Sample")
    ax.set_ylabel("Value")
    ax.legend()

    # Print the formatted validation table
    report = style.validate(fig)
    print(report)
    plt.close(fig)

## 6. Read Individual Check Results

The report object gives you access to each check result individually, which is useful if you want to loop over them, filter for failures, or save the results to a file.

- `report.passed`: `True` if the figure passed all checks
- `report.checks`: every check result
- `report.failures`: only the failed checks
- `report.to_dict()`: the full report as a plain dictionary (easy to save as JSON)

In [ ]:
print(f"Overall passed: {report.passed}")
print(f"Total checks: {len(report.checks)}\n")

# Iterate over every check result
icons = {"PASS": "pass", "FAIL": "FAIL", "WARN": "warn"}
for result in report.checks:
    icon = icons.get(result.status.value, "?")
    print(f"[{icon}] {result.check_name}: {result.message}")
    if result.fix_suggestion:
        print(f"       Fix: {result.fix_suggestion}")

# Serialize to a plain dict, ready for json.dumps() in a CI pipeline
report_dict = report.to_dict()
print(f"\nReport dict keys: {list(report_dict.keys())}")